# Common Table Expressions (CTEs)

## Overview
A **Common Table Expression (CTE)** — written with the `WITH` clause — is a named, reusable subquery defined at the top of a query. It exists only for the duration of the statement.

```sql
WITH cte_name AS (
    SELECT ...
),
second_cte AS (
    SELECT ...
    FROM   cte_name          -- CTEs can reference previously defined CTEs
)
SELECT *
FROM   second_cte
WHERE  ...;
```

**Advantages over derived tables:**

| | Derived table | CTE |
|---|---|---|
| Named and reusable within the query | No | Yes |
| Readable top-to-bottom | No (inside-out) | Yes |
| Can reference earlier CTEs | No | Yes |
| Supports recursion | No | Yes (see `recursive_ctes`) |
| Supported everywhere | Yes | SQLite 3.35+, all modern RDBMS |

**Are CTEs materialised?**
In PostgreSQL, CTEs are optionally materialised (the result is computed once and cached). In SQLite, CTEs are generally inlined (the optimiser expands them). For very large intermediate results, `CREATE TEMP TABLE` may be more efficient than a CTE.

**Multiple CTEs** are separated by commas — not by repeating `WITH`.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, active INTEGER DEFAULT 1);
CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT, province TEXT, manager_id INTEGER);
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_id INTEGER);
CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER, dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER);
CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT, category TEXT);
CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER, test_name TEXT, result_val REAL, units TEXT, collected TEXT, flagged INTEGER);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),
  (5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),
  (7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),
  (9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),
  (11,'Diana','Flores','2000-02-14','F','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),(11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Osei','Family Medicine','ON',10),(13,'Dr. Ethan Larson','Orthopaedics','QC',10),
  (14,'Dr. Priya Sharma','Emergency','AB',10),(15,'Dr. Lucas Petit','Radiology','QC',11);
INSERT INTO departments VALUES
  (1,'Emergency','Tower A',14),(2,'Cardiology','Tower B',10),(3,'Oncology','Tower C',11),
  (4,'Family Medicine','Clinic',12),(5,'Orthopaedics','Tower A',13),
  (6,'Radiology','Tower B',15),(7,'ICU','Tower C',NULL);
INSERT INTO encounters VALUES
  (1,1,10,2,'2023-01-15','I25.1',3200.00,1),(2,2,14,1,'2023-02-20','J18.9',1850.00,1),
  (3,3,12,4,'2023-03-05','Z00.0',120.00,0),(4,4,13,5,'2023-03-18','M17.1',5500.00,1),
  (5,5,12,4,'2023-04-02','J06.9',95.00,0),(6,6,14,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,2,'2023-06-22','I10',2100.00,1),(8,8,12,4,'2023-07-14',NULL,80.00,0),
  (9,1,14,1,'2023-08-30','R55',410.00,0),(10,3,12,4,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,2,'2023-10-01','I48.0',1750.00,1),(12,10,14,1,'2023-11-03','S52.5',2200.00,1),
  (13,2,10,2,'2023-11-20','I25.1',2900.00,1),(14,6,12,4,'2023-12-01','Z00.0',115.00,0);
INSERT INTO diagnoses VALUES
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),
  ('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),
  ('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),
  ('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');
INSERT INTO lab_results VALUES
  (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),
  (3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),
  (5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),
  (7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),
  (9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),
  (11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);
""")
conn.commit()
print('Healthcare schema ready.')


Healthcare schema ready.


---
## Single CTE — naming an intermediate result

In [2]:
# Replace a repeated scalar subquery with a single CTE
print('=== CTE: patient encounter summary, reused twice in the outer query ===')
q = """
WITH patient_summary AS (
    SELECT  patient_id,
            COUNT(*)                AS enc_count,
            ROUND(SUM(cost_cad),2)  AS total_cost,
            ROUND(AVG(cost_cad),2)  AS avg_cost,
            MAX(enc_date)           AS last_enc_date,
            SUM(admitted)           AS admissions
    FROM    encounters
    GROUP BY patient_id
)
SELECT  p.first_name || ' ' || p.last_name  AS patient,
        p.province,
        ps.enc_count,
        ps.total_cost,
        ps.avg_cost,
        ps.last_enc_date,
        ps.admissions,
        ROUND(100.0 * ps.admissions / ps.enc_count, 1) AS admission_rate_pct
FROM    patients       AS p
JOIN    patient_summary AS ps ON p.patient_id = ps.patient_id
ORDER BY ps.total_cost DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== CTE: patient encounter summary, reused twice in the outer query ===
         patient province  enc_count  total_cost  avg_cost last_enc_date  admissions  admission_rate_pct
   James MacLeod       NB          1      5500.0    5500.0    2023-03-18           1               100.0
       Liam Chen       NS          2      4750.0    2375.0    2023-11-20           2               100.0
     Aroha Ngata       NB          2      3610.0    1805.0    2023-08-30           1                50.0
   Marcus Okafor       ON          1      2200.0    2200.0    2023-11-03           1               100.0
       Mei Zhang       ON          1      2100.0    2100.0    2023-06-22           1               100.0
      Priya Nair       BC          1      1750.0    1750.0    2023-10-01           1               100.0
   Noah Williams       AB          2       895.0     447.5    2023-12-01           0                 0.0
Fatima Al-Rashid       ON          2       230.0     115.0    2023-09-12           0    

---
## Chaining multiple CTEs

In [3]:
# Multi-CTE pipeline: base → enrich → classify → final report
print('=== Multi-CTE pipeline: encounter data → enriched → tiered → summary ===')
q = """
WITH
-- Step 1: encounter costs with diagnosis info
enc_enriched AS (
    SELECT  e.enc_id,
            e.patient_id,
            e.enc_date,
            e.cost_cad,
            e.admitted,
            COALESCE(d.category, 'Uncoded')  AS diag_category
    FROM    encounters AS e
    LEFT JOIN diagnoses AS d ON e.diag_code = d.diag_code
),
-- Step 2: aggregate per patient
patient_stats AS (
    SELECT  patient_id,
            COUNT(*)                    AS enc_count,
            ROUND(SUM(cost_cad), 2)     AS total_cost,
            SUM(admitted)               AS admissions,
            COUNT(DISTINCT diag_category) AS categories_seen
    FROM    enc_enriched
    GROUP BY patient_id
),
-- Step 3: classify patients by cost tier
patient_tiered AS (
    SELECT  *,
            CASE
                WHEN total_cost < 500  THEN 'Low'
                WHEN total_cost < 2000 THEN 'Medium'
                ELSE                        'High'
            END AS cost_tier
    FROM    patient_stats
)
-- Final: summary by tier
SELECT  cost_tier,
        COUNT(*)                    AS patients,
        ROUND(AVG(total_cost), 2)   AS avg_total_cost,
        ROUND(AVG(enc_count), 1)    AS avg_encounters,
        SUM(admissions)             AS total_admissions
FROM    patient_tiered
GROUP BY cost_tier
ORDER BY avg_total_cost DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== Multi-CTE pipeline: encounter data → enriched → tiered → summary ===
cost_tier  patients  avg_total_cost  avg_encounters  total_admissions
     High         5          3632.0             1.4                 6
   Medium         2          1322.5             1.5                 1
      Low         3           135.0             1.3                 0


---
## Reusing a CTE multiple times in one query

In [4]:
# Reference the same CTE twice — avoids duplicating a complex subquery
print('=== CTE referenced twice: patient totals vs overall average ===')
q = """
WITH patient_costs AS (
    SELECT  patient_id,
            ROUND(SUM(cost_cad), 2)  AS total_cost
    FROM    encounters
    GROUP BY patient_id
),
overall AS (
    SELECT ROUND(AVG(total_cost), 2) AS avg_cost
    FROM   patient_costs   -- reuses patient_costs CTE
)
SELECT  p.first_name || ' ' || p.last_name    AS patient,
        pc.total_cost,
        o.avg_cost                             AS overall_avg,
        ROUND(pc.total_cost - o.avg_cost, 2)   AS diff,
        CASE WHEN pc.total_cost > o.avg_cost
             THEN 'Above avg' ELSE 'Below avg' END AS vs_avg
FROM    patient_costs  AS pc
JOIN    patients       AS p  ON pc.patient_id = p.patient_id
CROSS JOIN overall     AS o
ORDER BY pc.total_cost DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== CTE referenced twice: patient totals vs overall average ===
         patient  total_cost  overall_avg    diff    vs_avg
   James MacLeod      5500.0       2121.0  3379.0 Above avg
       Liam Chen      4750.0       2121.0  2629.0 Above avg
     Aroha Ngata      3610.0       2121.0  1489.0 Above avg
   Marcus Okafor      2200.0       2121.0    79.0 Above avg
       Mei Zhang      2100.0       2121.0   -21.0 Below avg
      Priya Nair      1750.0       2121.0  -371.0 Below avg
   Noah Williams       895.0       2121.0 -1226.0 Below avg
Fatima Al-Rashid       230.0       2121.0 -1891.0 Below avg
    Sofia Petrov        95.0       2121.0 -2026.0 Below avg
  Ethan Tremblay        80.0       2121.0 -2041.0 Below avg


---
## CTE vs derived table — readability comparison

In [5]:
# Same query written two ways — choose CTE for maintainability
print('=== Side-by-side: derived table vs CTE for the same logic ===')

derived = """
-- Derived table version (nested, hard to follow)
SELECT dept_name, enc_count, avg_cost
FROM (
    SELECT d.dept_name,
           COUNT(e.enc_id)           AS enc_count,
           ROUND(AVG(e.cost_cad),2)  AS avg_cost
    FROM   encounters AS e
    JOIN   departments AS d ON e.dept_id = d.dept_id
    GROUP BY d.dept_id, d.dept_name
) AS dept_summary
WHERE avg_cost > 1000
ORDER BY avg_cost DESC
"""

cte_version = """
-- CTE version (linear, named steps)
WITH dept_summary AS (
    SELECT d.dept_name,
           COUNT(e.enc_id)           AS enc_count,
           ROUND(AVG(e.cost_cad),2)  AS avg_cost
    FROM   encounters  AS e
    JOIN   departments AS d ON e.dept_id = d.dept_id
    GROUP BY d.dept_id, d.dept_name
)
SELECT dept_name, enc_count, avg_cost
FROM   dept_summary
WHERE  avg_cost > 1000
ORDER BY avg_cost DESC
"""

print('Derived table result:')
print(pd.read_sql(derived, conn).to_string(index=False))
print()
print('CTE result (identical):')
print(pd.read_sql(cte_version, conn).to_string(index=False))
print()
print('Both produce identical results. CTEs win on readability, especially')
print('when the intermediate step is complex or reused.')

conn.close()


=== Side-by-side: derived table vs CTE for the same logic ===
Derived table result:
   dept_name  enc_count  avg_cost
Orthopaedics          1    5500.0
  Cardiology          4    2487.5
   Emergency          4    1310.0

CTE result (identical):
   dept_name  enc_count  avg_cost
Orthopaedics          1    5500.0
  Cardiology          4    2487.5
   Emergency          4    1310.0

Both produce identical results. CTEs win on readability, especially
when the intermediate step is complex or reused.


---
## Common Pitfalls

**1. Using WITH multiple times instead of separating CTEs with commas**
`WITH cte1 AS (...) WITH cte2 AS (...)` is a syntax error. Multiple CTEs use a single `WITH` with comma-separated definitions: `WITH cte1 AS (...), cte2 AS (...)`.

**2. Referencing a CTE defined later in the same WITH block**
CTEs are evaluated in order. `cte2` can reference `cte1` only if `cte1` is defined first. Forward references are not allowed in standard SQL (except in recursive CTEs).

**3. Assuming CTEs are always materialised**
In PostgreSQL (pre-12), CTEs are materialised — computed once, result cached. In PostgreSQL 12+, the optimiser may inline them. In SQLite, CTEs are almost always inlined. If you need guaranteed materialisation for performance (e.g. avoiding repeated expensive computation), use `CREATE TEMP TABLE` or PostgreSQL's `MATERIALIZED` keyword: `WITH cte AS MATERIALIZED (...)`.

**4. CTE names that shadow table names**
`WITH encounters AS (SELECT ...)` — if the CTE is named `encounters`, any reference to `encounters` in the query body refers to the CTE, not the base table. This can produce surprising results if you intended to also query the base table. Use descriptive, distinct names.

**5. CTEs are scoped to a single statement only**
A CTE defined in one query is not visible to another query in the same session. If you need to reuse a complex query result across multiple statements, create a view or a temp table instead.

**6. Over-engineering with too many CTEs**
Five CTEs that each do one small transformation can be harder to follow than two well-named CTEs that do meaningful work. Group related transformations into a single CTE when they're always used together. Name CTEs by what they represent, not what they do: `patient_costs` not `step3_calculate_costs`.


---
*sql_methods_library - Samantha McGarrigle*